# Modules du pipeline — fichiers sources

Ce notebook regroupe **tous les fichiers `.py` du pipeline** en un seul fichier
facile à envoyer. Chaque cellule ci-dessous correspond à **un fichier**, et
utilise la magie Jupyter `%%writefile` : quand vous **exécutez une cellule**,
elle écrit le fichier `.py` correspondant sur le disque, dans le même dossier
que ce notebook.

**Mode d'emploi :**
1. Placez ce notebook (`modules.ipynb`) et `pipeline.ipynb` dans le même dossier.
2. Exécutez **toutes les cellules de ce notebook** (Run All) — cela recrée les
   5 fichiers `.py` nécessaires au pipeline.
3. Ouvrez et exécutez ensuite `pipeline.ipynb` normalement.

Vous pouvez aussi ignorer l'exécution et simplement **lire/copier le contenu**
de chaque cellule si vous préférez recréer les fichiers manuellement.

| Fichier | Rôle |
|---|---|
| `config.py` | Mots-clés EN/FR, poids et seuils. Modifiez les règles de détection ici. |
| `text_extraction.py` | Extraction de texte : natif (PDF texte), OCR (PDF scannés), hybride. Remplacez le moteur OCR ici. |
| `relevance.py` | Scoring de pertinence : mots-clés, densité numérique, zero-shot (optionnel), combinaison pondérée. |
| `pdf_processor.py` | Traite un PDF ou un dossier entier, écrit les PDF filtrés (pages pertinentes uniquement). |
| `report.py` | Génère le fichier Excel global (onglets Résumé + Détail_pages). |

### `config.py`

Mots-clés EN/FR, poids et seuils. Modifiez les règles de détection ici.

In [ ]:
%%writefile config.py
"""
config.py
---------
Configuration centrale du pipeline : mots-clés (EN/FR) et paramètres par défaut.

Toute modification des règles de détection (nouveaux mots-clés, nouveaux poids,
nouveaux seuils) se fait UNIQUEMENT ici, sans toucher au reste du code.
"""

# ---------------------------------------------------------------------------
# 1. MOTS-CLES / EXPRESSIONS A RECHERCHER
# ---------------------------------------------------------------------------
# Chaque statement financier est une liste de variantes (le mot "consolidated"
# / "consolidé" est ignoré volontairement, comme demandé dans les guidelines).

KEYWORDS_EN = {
    "income_statement": [
        "income statement",
        "profit and loss statement",
        "statement of earnings",
        "statement of operations",
        "statement of results",
    ],
    "balance_sheet": [
        "balance sheet",
        "statement of financial position",
        "statement of assets and liabilities",
        "statement of financial condition",
    ],
    "cash_flow_statement": [
        "cash flow statement",
        "statement of cash flows",
        "cash flow report",
        "statement of cash inflows and outflows",
    ],
    "statement_changes_equity": [
        "statement of changes in equity",
        "statement of shareholders equity",
        "statement of shareholders' equity",
        "statement of owners equity",
        "equity movement statement",
        "statement of equity",
    ],
    "statement_comprehensive_income": [
        "statement of comprehensive income",
        "comprehensive income statement",
        "statement of total comprehensive income",
        "statement of comprehensive earnings",
    ],
}

KEYWORDS_FR = {
    "compte_de_resultat": [
        "compte de resultat",
        "etat du resultat",
        "etat des resultats",
        "compte de pertes et profits",
        "etat des gains et pertes",
    ],
    "bilan": [
        "bilan",
        "bilan comptable",
        "etat de la situation financiere",
        "etat du bilan",
        "bilan des actifs et passifs",
    ],
    "tableau_flux_tresorerie": [
        "tableau des flux de tresorerie",
        "etat des flux de tresorerie",
        "flux de tresorerie",
        "etat des mouvements de tresorerie",
        "releve des flux de tresorerie",
    ],
    "tableau_variations_capitaux_propres": [
        "etat des variations des capitaux propres",
        "tableau des mouvements de capitaux propres",
        "etat des variations du patrimoine net",
        "etat des changements de capitaux propres",
        "tableau des variations de l'equite",
        "tableau des variations de l equite",
    ],
    "etat_resultat_global": [
        "etat du resultat global",
        "compte de resultat global",
        "etat du revenu global",
        "releve du resultat global",
        "tableau du resultat global",
    ],
}

# Mot à ignorer dans la comparaison (on le retire du texte avant de matcher,
# afin qu'il n'influence ni positivement ni négativement la détection).
IGNORED_WORDS = ["consolidated", "consolide", "consolidée", "consolidées", "consolidés"]


# ---------------------------------------------------------------------------
# 2. PARAMETRES PAR DEFAUT DU PIPELINE
# ---------------------------------------------------------------------------

DEFAULT_SETTINGS = {
    # --- OCR / extraction de texte ---
    "ocr_lang": "eng+fra",          # langues tesseract
    "ocr_dpi": 300,                  # résolution de rasterisation pour l'OCR
    # nombre minimal de caractères "utiles" extraits nativement en dessous
    # duquel on considère la page comme scannée et on bascule sur l'OCR
    "native_text_min_chars": 40,

    # --- scoring de pertinence ---
    # poids relatif de chaque scorer dans le score composite (somme libre,
    # ils sont normalisés automatiquement)
    "weight_keyword": 0.6,
    "weight_numeric_density": 0.25,
    "weight_zero_shot": 0.15,

    # seuil de densité numérique (proportion de caractères numériques /
    # caractères totaux) au-delà duquel on considère la page "riche en chiffres"
    "numeric_density_threshold": 0.06,

    # seuil final au-dessus duquel une page est jugée "pertinente"
    "relevance_threshold": 0.35,

    # activer/désactiver le classifieur zero-shot (nécessite `transformers`+`torch`)
    "use_zero_shot": False,
    "zero_shot_model": "joeddav/xlm-roberta-large-xnli",  # multilingue EN/FR
    "zero_shot_labels": [
        "financial statement page (income statement, balance sheet, cash flow, equity)",
        "irrelevant page (cover page, table of contents, notes, disclaimer, appendix)",
    ],
}


### `text_extraction.py`

Extraction de texte : natif (PDF texte), OCR (PDF scannés), hybride. Remplacez le moteur OCR ici.

In [ ]:
%%writefile text_extraction.py
"""
text_extraction.py
-------------------
Module responsable UNIQUEMENT d'extraire le texte de chaque page d'un PDF.

Architecture volontairement modulaire :
- `TextExtractor` est une interface abstraite. Le reste du pipeline (relevance,
  pdf_processor) ne dépend QUE de cette interface (méthode `extract_page_texts`).
- On peut donc remplacer le moteur OCR (Tesseract -> EasyOCR, PaddleOCR, une
  API cloud, etc.) en écrivant une nouvelle classe qui hérite de
  `TextExtractor`, SANS RIEN CHANGER ailleurs dans le pipeline.

Trois implémentations sont fournies :
1. NativeTextExtractor  : lit le texte déjà présent dans le PDF (pdfplumber).
   Rapide, mais renvoie une chaîne vide/quasi-vide sur un PDF scanné.
2. TesseractOCRExtractor: rasterise chaque page (pdf2image/poppler) puis
   applique un OCR (pytesseract/Tesseract). Fonctionne sur les PDF scannés.
3. HybridTextExtractor  : essaie d'abord l'extraction native page par page ;
   si le texte natif est trop court (page probablement scannée), bascule
   automatiquement sur l'OCR pour CETTE page uniquement. C'est l'extracteur
   recommandé à utiliser dans le pipeline principal.
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List

import pdfplumber
import pytesseract
from pdf2image import convert_from_path


@dataclass
class PageText:
    """Texte extrait d'une page, avec un indicateur de la méthode utilisée."""
    page_number: int          # 1-indexed
    text: str
    method: str               # "native" ou "ocr" (ou autre nom d'implémentation)


class TextExtractor(ABC):
    """Interface commune à tout moteur d'extraction de texte."""

    @abstractmethod
    def extract_page_texts(self, pdf_path: str) -> List[PageText]:
        """Retourne une liste de PageText, une entrée par page du PDF."""
        raise NotImplementedError


# ---------------------------------------------------------------------------
# 1. Extraction native (PDF "normaux", texte déjà encodé)
# ---------------------------------------------------------------------------
class NativeTextExtractor(TextExtractor):
    def extract_page_texts(self, pdf_path: str) -> List[PageText]:
        results: List[PageText] = []
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages, start=1):
                text = page.extract_text() or ""
                results.append(PageText(page_number=i, text=text, method="native"))
        return results


# ---------------------------------------------------------------------------
# 2. Extraction OCR (PDF scannés / images)
#    -> C'est CETTE classe qu'il faut remplacer si vous changez de moteur OCR
#       (ex: EasyOCR, PaddleOCR, Google Vision, Azure Document Intelligence...).
#       Il suffit de respecter l'interface TextExtractor.
# ---------------------------------------------------------------------------
class TesseractOCRExtractor(TextExtractor):
    def __init__(self, lang: str = "eng+fra", dpi: int = 300):
        self.lang = lang
        self.dpi = dpi

    def extract_page_texts(self, pdf_path: str) -> List[PageText]:
        images = convert_from_path(pdf_path, dpi=self.dpi)
        results: List[PageText] = []
        for i, img in enumerate(images, start=1):
            text = pytesseract.image_to_string(img, lang=self.lang)
            results.append(PageText(page_number=i, text=text, method="ocr"))
        return results

    def extract_single_page_text(self, pdf_path: str, page_number: int) -> str:
        """OCR d'une seule page (1-indexed) - utilisé par HybridTextExtractor
        pour éviter de re-rasteriser tout le PDF quand une seule page a besoin d'OCR."""
        images = convert_from_path(
            pdf_path, dpi=self.dpi, first_page=page_number, last_page=page_number
        )
        if not images:
            return ""
        return pytesseract.image_to_string(images[0], lang=self.lang)


# ---------------------------------------------------------------------------
# 3. Extracteur hybride recommandé : natif d'abord, OCR en secours page par page
# ---------------------------------------------------------------------------
class HybridTextExtractor(TextExtractor):
    def __init__(
        self,
        native_extractor: TextExtractor | None = None,
        ocr_extractor: TesseractOCRExtractor | None = None,
        native_text_min_chars: int = 40,
    ):
        self.native_extractor = native_extractor or NativeTextExtractor()
        self.ocr_extractor = ocr_extractor or TesseractOCRExtractor()
        self.native_text_min_chars = native_text_min_chars

    def extract_page_texts(self, pdf_path: str) -> List[PageText]:
        native_pages = self.native_extractor.extract_page_texts(pdf_path)
        final_pages: List[PageText] = []

        for page in native_pages:
            if len(page.text.strip()) >= self.native_text_min_chars:
                final_pages.append(page)
            else:
                # Page probablement scannée -> OCR ciblé sur cette page
                ocr_text = self.ocr_extractor.extract_single_page_text(
                    pdf_path, page.page_number
                )
                final_pages.append(
                    PageText(page_number=page.page_number, text=ocr_text, method="ocr")
                )
        return final_pages


### `relevance.py`

Scoring de pertinence : mots-clés, densité numérique, zero-shot (optionnel), combinaison pondérée.

In [ ]:
%%writefile relevance.py
"""
relevance.py
------------
Module responsable UNIQUEMENT de juger si le texte d'une page est "pertinent"
(page de statement financier) ou non.

Comme pour l'extraction, chaque méthode de scoring est une classe qui respecte
l'interface `RelevanceScorer` (méthode `score(text) -> float` dans [0, 1]).
Le `CompositeScorer` les combine par une moyenne pondérée (poids définis dans
config.py). On peut activer/désactiver ou remplacer un scorer sans toucher
au reste du pipeline.
"""

from __future__ import annotations

import re
import unicodedata
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Dict, List, Optional

from config import (
    KEYWORDS_EN,
    KEYWORDS_FR,
    IGNORED_WORDS,
    DEFAULT_SETTINGS,
)


def _normalize(text: str) -> str:
    """Minuscule, sans accents, espaces multiples réduits - pour un matching robuste
    (OCR renvoie parfois des accents mal reconnus)."""
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    for w in IGNORED_WORDS:
        w_norm = unicodedata.normalize("NFKD", w.lower())
        w_norm = "".join(c for c in w_norm if not unicodedata.combining(c))
        text = text.replace(w_norm, " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


@dataclass
class RelevanceResult:
    score: float
    is_relevant: bool
    matched_keywords: List[str] = field(default_factory=list)
    details: Dict[str, float] = field(default_factory=dict)


class RelevanceScorer(ABC):
    @abstractmethod
    def score(self, text: str) -> float:
        """Retourne un score dans [0, 1]."""
        raise NotImplementedError


# ---------------------------------------------------------------------------
# 1. Scorer par mots-clés (règle 1 des guidelines)
# ---------------------------------------------------------------------------
class KeywordScorer(RelevanceScorer):
    def __init__(self, keywords_en: Dict[str, List[str]] = None, keywords_fr: Dict[str, List[str]] = None):
        self.keywords_en = keywords_en or KEYWORDS_EN
        self.keywords_fr = keywords_fr or KEYWORDS_FR
        # à plat, normalisées une fois pour toutes
        self._all_terms = []
        for group in list(self.keywords_en.values()) + list(self.keywords_fr.values()):
            for term in group:
                self._all_terms.append(_normalize(term))

    def matched_terms(self, text: str) -> List[str]:
        norm_text = _normalize(text)
        return [term for term in self._all_terms if term in norm_text]

    def score(self, text: str) -> float:
        # dès qu'un terme matche, la page est considérée pleinement pertinente
        # sur ce critère (score binaire 0/1), conformément à la règle
        # "si il y a un match, ca garde la page".
        return 1.0 if self.matched_terms(text) else 0.0


# ---------------------------------------------------------------------------
# 2. Scorer de densité numérique (règle 2 des guidelines)
# ---------------------------------------------------------------------------
class NumericDensityScorer(RelevanceScorer):
    def __init__(self, threshold: float = 0.06):
        self.threshold = threshold

    def density(self, text: str) -> float:
        if not text:
            return 0.0
        digit_count = sum(c.isdigit() for c in text)
        return digit_count / max(len(text), 1)

    def score(self, text: str) -> float:
        d = self.density(text)
        # normalisation douce : au seuil -> 1.0, en dessous -> proportionnel
        return min(d / self.threshold, 1.0) if self.threshold > 0 else 0.0


# ---------------------------------------------------------------------------
# 3. Scorer zero-shot (règle 3, optionnel - nécessite `transformers` + `torch`)
# ---------------------------------------------------------------------------
class ZeroShotScorer(RelevanceScorer):
    """Scorer optionnel basé sur un modèle de classification zero-shot local
    (Hugging Face). Désactivé par défaut (use_zero_shot=False dans config.py)
    car il nécessite `pip install transformers torch` et un téléchargement de
    modèle. Peut être remplacé par n'importe quel modèle zero-shot compatible
    `pipeline("zero-shot-classification", ...)`.
    """

    def __init__(self, model_name: str, candidate_labels: List[str]):
        try:
            from transformers import pipeline
        except ImportError as e:
            raise ImportError(
                "ZeroShotScorer nécessite `transformers` et `torch`. "
                "Installez-les avec: pip install transformers torch"
            ) from e
        self._classifier = pipeline("zero-shot-classification", model=model_name)
        self.candidate_labels = candidate_labels
        # on suppose que le premier label de la liste = label "pertinent"
        self.relevant_label = candidate_labels[0]

    def score(self, text: str) -> float:
        if not text.strip():
            return 0.0
        # on tronque le texte pour rester dans la limite de tokens du modèle
        truncated = text[:2000]
        result = self._classifier(truncated, self.candidate_labels)
        label_scores = dict(zip(result["labels"], result["scores"]))
        return float(label_scores.get(self.relevant_label, 0.0))


# ---------------------------------------------------------------------------
# 4. Scorer composite : combine les scorers ci-dessus avec des poids
# ---------------------------------------------------------------------------
class CompositeScorer:
    def __init__(
        self,
        keyword_scorer: KeywordScorer,
        numeric_scorer: NumericDensityScorer,
        zero_shot_scorer: Optional[ZeroShotScorer] = None,
        weight_keyword: float = 0.6,
        weight_numeric: float = 0.25,
        weight_zero_shot: float = 0.15,
        relevance_threshold: float = 0.35,
        keyword_auto_accept: bool = True,
    ):
        self.keyword_scorer = keyword_scorer
        self.numeric_scorer = numeric_scorer
        self.zero_shot_scorer = zero_shot_scorer
        self.relevance_threshold = relevance_threshold
        # "auto-accept": si un mot-clé matche, la page est gardée directement,
        # peu importe les autres scores (comportement demandé initialement).
        self.keyword_auto_accept = keyword_auto_accept

        # normalisation des poids (ignore le poids zero-shot si le scorer est absent)
        w_zs = weight_zero_shot if zero_shot_scorer is not None else 0.0
        total = weight_keyword + weight_numeric + w_zs
        total = total if total > 0 else 1.0
        self.weight_keyword = weight_keyword / total
        self.weight_numeric = weight_numeric / total
        self.weight_zero_shot = w_zs / total

    def evaluate(self, text: str) -> RelevanceResult:
        matched = self.keyword_scorer.matched_terms(text)
        kw_score = 1.0 if matched else 0.0
        num_score = self.numeric_scorer.score(text)
        zs_score = self.zero_shot_scorer.score(text) if self.zero_shot_scorer else 0.0

        composite = (
            self.weight_keyword * kw_score
            + self.weight_numeric * num_score
            + self.weight_zero_shot * zs_score
        )

        if self.keyword_auto_accept and matched:
            is_relevant = True
        else:
            is_relevant = composite >= self.relevance_threshold

        return RelevanceResult(
            score=composite,
            is_relevant=is_relevant,
            matched_keywords=matched,
            details={
                "keyword_score": kw_score,
                "numeric_density_score": num_score,
                "zero_shot_score": zs_score,
            },
        )


def build_default_composite_scorer(settings: Dict = None) -> CompositeScorer:
    """Factory pratique : construit un CompositeScorer à partir de config.DEFAULT_SETTINGS
    (ou d'un dict de settings personnalisé)."""
    s = settings or DEFAULT_SETTINGS
    keyword_scorer = KeywordScorer()
    numeric_scorer = NumericDensityScorer(threshold=s["numeric_density_threshold"])

    zero_shot_scorer = None
    if s.get("use_zero_shot"):
        zero_shot_scorer = ZeroShotScorer(
            model_name=s["zero_shot_model"],
            candidate_labels=s["zero_shot_labels"],
        )

    return CompositeScorer(
        keyword_scorer=keyword_scorer,
        numeric_scorer=numeric_scorer,
        zero_shot_scorer=zero_shot_scorer,
        weight_keyword=s["weight_keyword"],
        weight_numeric=s["weight_numeric_density"],
        weight_zero_shot=s["weight_zero_shot"],
        relevance_threshold=s["relevance_threshold"],
    )


### `pdf_processor.py`

Traite un PDF ou un dossier entier, écrit les PDF filtrés (pages pertinentes uniquement).

In [ ]:
%%writefile pdf_processor.py
"""
pdf_processor.py
----------------
Orchestration au niveau d'UN SEUL PDF :
1. Extraire le texte de chaque page (via un TextExtractor, ex: HybridTextExtractor)
2. Juger la pertinence de chaque page (via un CompositeScorer)
3. Écrire un nouveau PDF ne contenant que les pages jugées pertinentes

Ce module ne connaît ni le moteur OCR précis, ni le détail des règles de
scoring : il dépend uniquement des interfaces `TextExtractor` et
`CompositeScorer`. On peut donc changer d'OCR ou de logique de scoring sans
modifier une seule ligne ici.
"""

from __future__ import annotations

import os
from dataclasses import dataclass, field
from typing import List

from pypdf import PdfReader, PdfWriter

from text_extraction import TextExtractor
from relevance import CompositeScorer, RelevanceResult


@dataclass
class PageResult:
    page_number: int           # 1-indexed
    is_relevant: bool
    score: float
    matched_keywords: List[str]
    extraction_method: str


@dataclass
class PdfProcessingResult:
    pdf_name: str
    input_path: str
    output_path: str | None
    total_pages: int
    relevant_pages: List[int] = field(default_factory=list)
    page_results: List[PageResult] = field(default_factory=list)
    error: str | None = None


def process_single_pdf(
    pdf_path: str,
    extractor: TextExtractor,
    scorer: CompositeScorer,
    output_dir: str,
) -> PdfProcessingResult:
    """Traite un seul PDF: évalue chaque page et écrit le PDF filtré dans output_dir.

    Si aucune page n'est jugée pertinente, aucun fichier de sortie n'est écrit
    (output_path restera None) mais le résultat est quand même reporté (utile
    pour le suivi dans le rapport Excel).
    """
    pdf_name = os.path.basename(pdf_path)
    result = PdfProcessingResult(
        pdf_name=pdf_name, input_path=pdf_path, output_path=None, total_pages=0
    )

    try:
        page_texts = extractor.extract_page_texts(pdf_path)
        result.total_pages = len(page_texts)

        reader = PdfReader(pdf_path)
        writer = PdfWriter()

        for page_text in page_texts:
            evaluation: RelevanceResult = scorer.evaluate(page_text.text)

            result.page_results.append(
                PageResult(
                    page_number=page_text.page_number,
                    is_relevant=evaluation.is_relevant,
                    score=evaluation.score,
                    matched_keywords=evaluation.matched_keywords,
                    extraction_method=page_text.method,
                )
            )

            if evaluation.is_relevant:
                result.relevant_pages.append(page_text.page_number)
                writer.add_page(reader.pages[page_text.page_number - 1])

        if result.relevant_pages:
            os.makedirs(output_dir, exist_ok=True)
            output_path = os.path.join(output_dir, pdf_name)
            with open(output_path, "wb") as f:
                writer.write(f)
            result.output_path = output_path

    except Exception as e:  # on isole l'erreur pour ne pas casser le traitement du dossier entier
        result.error = f"{type(e).__name__}: {e}"

    return result


def process_folder(
    input_dir: str,
    output_dir: str,
    extractor: TextExtractor,
    scorer: CompositeScorer,
    verbose: bool = True,
) -> List[PdfProcessingResult]:
    """Traite tous les PDF d'un dossier et retourne la liste des résultats
    (un PdfProcessingResult par fichier)."""
    os.makedirs(output_dir, exist_ok=True)

    pdf_files = sorted(
        f for f in os.listdir(input_dir) if f.lower().endswith(".pdf")
    )

    results: List[PdfProcessingResult] = []
    for i, filename in enumerate(pdf_files, start=1):
        pdf_path = os.path.join(input_dir, filename)
        if verbose:
            print(f"[{i}/{len(pdf_files)}] Traitement de: {filename}")

        res = process_single_pdf(pdf_path, extractor, scorer, output_dir)
        results.append(res)

        if verbose:
            if res.error:
                print(f"    -> ERREUR: {res.error}")
            else:
                print(
                    f"    -> {len(res.relevant_pages)}/{res.total_pages} "
                    f"page(s) pertinente(s): {res.relevant_pages}"
                )

    return results


### `report.py`

Génère le fichier Excel global (onglets Résumé + Détail_pages).

In [ ]:
%%writefile report.py
"""
report.py
---------
Génère le fichier Excel global récapitulant, pour chaque PDF traité,
les pages jugées pertinentes.

Deux feuilles sont produites :
- "Résumé"       : une ligne par PDF (nom, nb pages total, pages pertinentes, etc.)
- "Détail_pages" : une ligne par page traitée (score, mots-clés trouvés, méthode
                    d'extraction) -> utile pour auditer / ajuster les seuils.
"""

from __future__ import annotations

from typing import List

import pandas as pd

from pdf_processor import PdfProcessingResult


def _format_pages_list(pages: List[int]) -> str:
    return ", ".join(str(p) for p in pages) if pages else ""


def build_summary_dataframe(results: List[PdfProcessingResult]) -> pd.DataFrame:
    rows = []
    for r in results:
        rows.append(
            {
                "Nom du PDF": r.pdf_name,
                "Chemin source": r.input_path,
                "Chemin PDF filtré": r.output_path or "",
                "Nombre de pages total": r.total_pages,
                "Nombre de pages pertinentes": len(r.relevant_pages),
                "Pages pertinentes (numéros)": _format_pages_list(r.relevant_pages),
                "Statut": "Erreur" if r.error else (
                    "Aucune page pertinente" if not r.relevant_pages else "OK"
                ),
                "Erreur": r.error or "",
            }
        )
    return pd.DataFrame(rows)


def build_detail_dataframe(results: List[PdfProcessingResult]) -> pd.DataFrame:
    rows = []
    for r in results:
        for pr in r.page_results:
            rows.append(
                {
                    "Nom du PDF": r.pdf_name,
                    "Page": pr.page_number,
                    "Pertinente": pr.is_relevant,
                    "Score composite": round(pr.score, 3),
                    "Mots-clés trouvés": ", ".join(pr.matched_keywords),
                    "Méthode d'extraction": pr.extraction_method,
                }
            )
    return pd.DataFrame(rows)


def write_excel_report(results: List[PdfProcessingResult], excel_path: str) -> None:
    summary_df = build_summary_dataframe(results)
    detail_df = build_detail_dataframe(results)

    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="Résumé", index=False)
        detail_df.to_excel(writer, sheet_name="Détail_pages", index=False)

        # ajustement simple de la largeur des colonnes pour la lisibilité
        for sheet_name, df in [("Résumé", summary_df), ("Détail_pages", detail_df)]:
            ws = writer.sheets[sheet_name]
            for idx, col in enumerate(df.columns, start=1):
                max_len = max(
                    [len(str(col))] + [len(str(v)) for v in df[col].astype(str)]
                )
                ws.column_dimensions[ws.cell(row=1, column=idx).column_letter].width = min(
                    max(max_len + 2, 12), 60
                )

    print(f"Rapport Excel écrit: {excel_path}")


---
### Vérification

Exécutez la cellule ci-dessous pour vérifier que les 5 fichiers ont bien été
écrits dans le dossier courant.

In [ ]:
import os
expected = ["config.py", "text_extraction.py", "relevance.py", "pdf_processor.py", "report.py"]
for f in expected:
    status = "OK" if os.path.exists(f) else "MANQUANT"
    print(f"{f:<25} {status}")
